In [ ]:
!pip install chromadb

In [1]:
import chromadb as cdb
from chromadb.config import Settings
import pandas as pd

In [2]:
df = pd.read_csv('medium_posts_kaggle_style_7500.csv')
df.head()

,title,subtitle,subtitle_truncated_flag,category,author,author_id,published_date,reading_time_minutes,word_count,claps,voters,responses,url,tags
0,Ultimate Guide to Sql #1,"Learn sql with real-world examples, best pract...",False,sql,Alice Johnson,1380,2023-07-18,10,1514,9144,839,346,https://medium.com/@demo/post-1,career;databricks;machine-learning
1,Hands-on Guide to Snowflake #2,"Learn snowflake with real-world examples, best...",False,snowflake,Alice Johnson,1016,2022-07-11,9,1552,33118,4931,13,https://medium.com/@demo/post-2,databricks;programming;career
2,Complete Guide to Cloud Computing #3,Learn cloud computing with real-world examples...,False,cloud-computing,Michael Chen,1302,2023-07-24,3,1253,45753,3462,174,https://medium.com/@demo/post-3,software-engineering;data-science;programming
3,Ultimate Guide to Python #4,"Learn python with real-world examples, best pr...",False,python,Bob Smith,1195,2022-07-18,14,2008,39565,2166,413,https://medium.com/@demo/post-4,artificial-intelligence;big-data;databricks
4,Hands-on Guide to Machine Learning #5,Learn machine learning with real-world example...,False,machine-learning,Bob Smith,1283,2023-08-24,14,2964,12601,569,23,https://medium.com/@demo/post-5,sql;programming;software-engineering


In [3]:
df = df.dropna()

df = df[~df['subtitle_truncated_flag']]

In [4]:
topic_of_learning = ["python", "cloud-computing", "data-science"]
df = df[df['category'].isin(topic_of_learning)]
df['text'] = df['title'] + " " + df['subtitle']
df['meta'] = df.apply(lambda x: {"text": x['text'], "category": x['category']}, axis=1)

In [5]:
df.head()

,title,subtitle,subtitle_truncated_flag,category,author,author_id,published_date,reading_time_minutes,word_count,claps,voters,responses,url,tags,text,meta
2,Complete Guide to Cloud Computing #3,Learn cloud computing with real-world examples...,False,cloud-computing,Michael Chen,1302,2023-07-24,3,1253,45753,3462,174,https://medium.com/@demo/post-3,software-engineering;data-science;programming,Complete Guide to Cloud Computing #3 Learn clo...,{'text': 'Complete Guide to Cloud Computing #3...
3,Ultimate Guide to Python #4,"Learn python with real-world examples, best pr...",False,python,Bob Smith,1195,2022-07-18,14,2008,39565,2166,413,https://medium.com/@demo/post-4,artificial-intelligence;big-data;databricks,Ultimate Guide to Python #4 Learn python with ...,{'text': 'Ultimate Guide to Python #4 Learn py...
15,Advanced Guide to Data Science #16,"Learn data science with real-world examples, b...",False,data-science,Michael Chen,1485,2025-01-30,8,1685,34581,4969,216,https://medium.com/@demo/post-16,programming;databricks;career,Advanced Guide to Data Science #16 Learn data ...,{'text': 'Advanced Guide to Data Science #16 L...
19,Complete Guide to Cloud Computing #20,Learn cloud computing with real-world examples...,False,cloud-computing,Bob Smith,1050,2024-06-01,14,2334,26941,3825,442,https://medium.com/@demo/post-20,career;artificial-intelligence;machine-learning,Complete Guide to Cloud Computing #20 Learn cl...,{'text': 'Complete Guide to Cloud Computing #2...
30,Practical Guide to Data Science #31,"Learn data science with real-world examples, b...",False,data-science,Emma Wilson,1310,2023-03-08,13,1433,45055,2162,258,https://medium.com/@demo/post-31,big-data;software-engineering;artificial-intel...,Practical Guide to Data Science #31 Learn data...,{'text': 'Practical Guide to Data Science #31 ...


In [9]:
# chroma db setup
client = cdb.Client(Settings(persist_directory="medium-chroma_db")) # in memory database if you dont specify persist_directory
# create a collection
collection_1 = client.create_collection(name="medium_posts_1")

InternalError: Collection [medium_posts_1] already exists

In [ ]:
# deleting the collection if it already exists
#collection.delete()

In [ ]:
collection_1.upsert(
    ids=[str(x) for x in df.index.tolist()],
    documents=df['text'].tolist(),
    metadatas=df['meta'].tolist()
)

In [ ]:
# has upsert operation, so if the id already exists, it will update the document and metadata using the id

In [ ]:
# vector query

query_str = """
what are the cloud offertings right now ?
"""

collection_1.query(
    query_texts=query_str,
    n_results=1
)

{'ids': [['1079']],
 'embeddings': None,
 'documents': [['Hands-on Guide to Cloud Computing #1080 Learn cloud computing with real-world examples, best practices, and production-ready techniques.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'cloud-computing',
    'text': 'Hands-on Guide to Cloud Computing #1080 Learn cloud computing with real-world examples, best practices, and production-ready techniques.'}]],
 'distances': [[1.0544723272323608]]}